# SBI812 — Lecture 7
## Exploratory Data Analysis with pandas, Matplotlib and Seaborn

### Learning outcomes

By the end of this lecture, a student should be able to:

1. import a CSV file into pandas;
2. identify rows, columns, data types and missing values;
3. clean simple text inconsistencies, missing values and duplicate rows;
4. select, filter and sort observations;
5. create useful summary tables with `value_counts()` and `groupby()`;
6. create histograms, count plots, box plots and scatter plots; and
7. interpret a table or figure in words connected to the research question.

This lecture follows a realistic data-science sequence:

```text
Import → inspect → clean → select → summarise → visualise → interpret
```

The code is written in small steps. Each new pandas, Matplotlib or Seaborn instruction is explained before it is combined with other instructions.

## 1. The plant-growth experiment

Researchers grew Bean and Maize plants under three nitrogen treatments. Ten plants were assigned to each species–treatment combination.

| Column | Meaning | Variable type |
|---|---|---|
| `sample_id` | Unique plant identifier | Identifier |
| `species` | Bean or Maize | Categorical |
| `treatment` | Control, Low_N or High_N | Categorical |
| `replicate` | Replicate number within a group | Integer |
| `temperature_c` | Growth temperature in °C | Numerical |
| `gene_expression` | Relative expression of a nitrogen-response gene | Numerical |
| `biomass_g` | Final dry biomass in grams | Numerical |
| `chlorophyll_index` | Leaf chlorophyll reading | Numerical |
| `leaf_count` | Number of leaves | Integer |
| `plant_status` | Healthy or Stressed | Categorical |

The supplied CSV is deliberately imperfect. It contains inconsistent text, missing measurements and one duplicated row. These are common features of real research data.

## 2. Import the required libraries

A **library** is a collection of code written for particular tasks.

- `pandas` supplies DataFrames and data-manipulation tools.
- `matplotlib.pyplot` supplies general plotting instructions.
- `seaborn` builds statistical figures using pandas DataFrames.

The names `pd`, `plt` and `sns` are conventional short aliases.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


`sns.set_theme(style="whitegrid")` changes the appearance of later figures. It does not change the data.

## 3. Read the CSV file

Place the notebook and `SBI812_plant_growth_experiment.csv` in the same directory. A relative path can then contain only the filename.

`pd.read_csv()` reads the file and returns a DataFrame. The variable name `raw_data` reminds us that cleaning has not yet been completed.

In [ ]:
raw_data = pd.read_csv("SBI812_plant_growth_experiment.csv")
raw_data.head()


`head()` displays the first five rows by default. It is an inspection method; it does not remove the other rows.

Parentheses are required because `head` is a method that pandas must execute.

## 4. Establish the size and structure

`shape` returns two numbers:

```text
(number of rows, number of columns)
```

Unlike `head()`, `shape` is an attribute and therefore has no parentheses.

In [ ]:
print("Rows and columns:", raw_data.shape)
print("Column names:")
print(raw_data.columns)


The raw file contains 61 rows because one row was accidentally duplicated. It contains 10 columns.

`columns` shows the exact spelling of every column name. This is useful because pandas requires column names to be written accurately.

## 5. Examine data types and missing values

`info()` reports:

- the column names;
- the number of non-missing values;
- each column's data type; and
- approximate memory use.

Common pandas data types include:

- `object`: usually text;
- `int64`: whole numbers;
- `float64`: numbers that may contain decimal points.

In [ ]:
raw_data.info()


A numerical column containing missing values is often stored as `float64`, even when most visible values are whole numbers.

To count missing values explicitly, use `isna()` followed by `sum()`:

1. `raw_data.isna()` creates a True/False table;
2. True means that a value is missing;
3. `sum()` counts the True values in each column.

In [ ]:
missing_counts = raw_data.isna().sum()
missing_counts


## 6. Inspect categorical values before cleaning

`value_counts()` counts how often each distinct value occurs. It can reveal spelling, capitalisation and spacing inconsistencies.

Start with the treatment column.

In [ ]:
raw_data["treatment"].value_counts(dropna=False)


Values such as `Control`, ` control ` and `HIGH_N` may describe the same intended categories but pandas treats them as different strings.

## 7. Create a working copy

Cleaning should not immediately overwrite the original DataFrame. `copy()` creates a separate working DataFrame.

In [ ]:
clean_data = raw_data.copy()


## 8. Clean text columns step by step

Consider this expression:

```python
clean_data["treatment"].str.strip()
```

- `clean_data["treatment"]` selects the treatment column.
- `.str` tells pandas that the following operation should be applied as a string operation to every value in the column.
- `.strip()` removes spaces at the beginning and end of every text value.

The next operation, `.str.title()`, gives values consistent capitalisation. Because an underscore separates words, `High_N` remains easy to read.

In [ ]:
treatment_without_spaces = clean_data["treatment"].str.strip()
clean_treatment = treatment_without_spaces.str.title()
clean_data["treatment"] = clean_treatment

species_without_spaces = clean_data["species"].str.strip()
clean_species = species_without_spaces.str.title()
clean_data["species"] = clean_species

print(clean_data["treatment"].value_counts())
print()
print(clean_data["species"].value_counts())


The intermediate variable names make every transformation visible. A shorter method chain is possible, but beginners should first understand what each method contributes.

## 9. Find the duplicated row

`duplicated()` compares complete rows. It returns `True` for an additional copy of a row already encountered.

In [ ]:
duplicate_rows = clean_data.duplicated()
print("Number of duplicated rows:", duplicate_rows.sum())

clean_data[duplicate_rows]


`drop_duplicates()` keeps one copy and removes the additional identical row. The result is assigned back to `clean_data`.

In [ ]:
clean_data = clean_data.drop_duplicates()
print("Rows after duplicate removal:", len(clean_data))


## 10. Handle missing measurements

There is no single correct treatment for every missing value. The decision depends on why the value is missing and how the column will be used.

For this teaching dataset:

- missing `gene_expression` and `chlorophyll_index` values are replaced with the median of their columns;
- the row with missing `biomass_g` is removed because biomass is the main outcome used later.

The median is less affected by unusually high or low values than the mean.

In [ ]:
gene_median = clean_data["gene_expression"].median()
chlorophyll_median = clean_data["chlorophyll_index"].median()

clean_data["gene_expression"] = clean_data["gene_expression"].fillna(gene_median)
clean_data["chlorophyll_index"] = clean_data["chlorophyll_index"].fillna(chlorophyll_median)

clean_data = clean_data.dropna(subset=["biomass_g"])

print(clean_data.isna().sum())
print("Rows remaining:", len(clean_data))


`fillna(value)` replaces missing entries with the supplied value.

`dropna(subset=["biomass_g"])` examines only the biomass column when deciding which rows to remove.

The cleaning choices should always be reported. Quietly replacing or deleting measurements can make an analysis impossible to reproduce.

## 11. Select columns

Single square brackets select one column as a Series. Double square brackets select several columns and return a DataFrame.

In [ ]:
biomass_series = clean_data["biomass_g"]

selected_table = clean_data[[
    "sample_id",
    "species",
    "treatment",
    "biomass_g"
]]

selected_table.head()


## 12. Filter rows using a condition

The following analysis selects only High_N plants.

First create the condition separately. This allows the True/False result to be inspected before it is used.

In [ ]:
is_high_n = clean_data["treatment"] == "High_N"
high_n_plants = clean_data[is_high_n]

high_n_plants.head()


The expression `==` asks whether two values are equal. It does not assign a new value.

Two conditions can be combined with `&`, meaning **and**. Each condition must be enclosed in parentheses.

In [ ]:
is_high_n = clean_data["treatment"] == "High_N"
is_maize = clean_data["species"] == "Maize"

high_n_maize = clean_data[is_high_n & is_maize]
high_n_maize[["sample_id", "biomass_g", "gene_expression"]]


## 13. Sort observations

`sort_values()` rearranges rows according to one or more columns. It does not alter the measured values.

`ascending=False` places the largest biomass first.

In [ ]:
largest_plants = clean_data.sort_values(
    by="biomass_g",
    ascending=False
)

largest_plants[["sample_id", "species", "treatment", "biomass_g"]].head(10)


## 14. Create a calculated column

A calculated column should represent a quantity with a clear meaning. Here, biomass per leaf is calculated as:

$$\text{biomass per leaf}=\frac{\text{biomass in grams}}{\text{number of leaves}}$$

pandas performs the division row by row.

In [ ]:
clean_data["biomass_per_leaf"] = (
    clean_data["biomass_g"] / clean_data["leaf_count"]
)

clean_data[["sample_id", "biomass_g", "leaf_count", "biomass_per_leaf"]].head()


## 15. Obtain descriptive statistics

`describe()` summarises numerical columns. Its output includes:

- `count`: non-missing observations;
- `mean`: arithmetic average;
- `std`: standard deviation;
- `min` and `max`: smallest and largest values;
- `25%`, `50%` and `75%`: quartiles.

The 50% value is the median.

In [ ]:
clean_data[[
    "gene_expression",
    "biomass_g",
    "chlorophyll_index"
]].describe()


## 16. Summarise categorical variables

`value_counts()` is appropriate when the question concerns category frequencies.

In [ ]:
status_counts = clean_data["plant_status"].value_counts()
status_counts


## 17. Calculate group summaries

`groupby()` temporarily divides the DataFrame into groups. An aggregation such as `mean()` is then calculated separately within each group.

The following calculation asks:

> What is the mean biomass for each treatment?

In [ ]:
treatment_groups = clean_data.groupby("treatment")
mean_biomass = treatment_groups["biomass_g"].mean()
mean_biomass


To compare species and treatment simultaneously, pass both column names as a list.

In [ ]:
species_treatment_summary = clean_data.groupby(
    ["species", "treatment"]
)[["gene_expression", "biomass_g"]].mean()

species_treatment_summary.round(2)


`round(2)` changes only the displayed number of decimal places. It does not change the underlying measurements.

## 18. Why visualisation is part of data analysis

A summary statistic compresses information into a number. A figure can reveal distribution shape, unusual observations, group overlap and relationships that one number may conceal.

The correct graph depends on the question:

| Question | Suitable graph |
|---|---|
| How is one numerical variable distributed? | Histogram |
| How many observations belong to each category? | Count plot |
| How does a numerical variable differ among categories? | Box plot |
| Are two numerical variables related? | Scatter plot |

## 19. Histogram with Matplotlib

A histogram divides a numerical variable into intervals called bins and counts observations in each interval.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(
    clean_data["biomass_g"],
    bins=10,
    color="steelblue",
    edgecolor="black"
)
plt.xlabel("Biomass (g)")
plt.ylabel("Number of plants")
plt.title("Distribution of plant biomass")
plt.tight_layout()
plt.show()


- `figure(figsize=(8, 5))` creates a plotting area eight inches wide and five inches high.
- `bins=10` requests ten intervals.
- `xlabel()`, `ylabel()` and `title()` make the graph understandable without reading the code.
- `tight_layout()` reduces accidental clipping of labels.
- `show()` displays the completed figure.

Do not describe a histogram only as “going up and down.” Discuss the centre, spread, shape and possible unusual observations.

## 20. Count plot with Seaborn

Seaborn can refer to DataFrame columns by name. `data=clean_data` identifies the DataFrame and `x="plant_status"` identifies the categorical column.

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(
    data=clean_data,
    x="plant_status",
    hue="plant_status",
    palette="Set2",
    legend=False
)
plt.xlabel("Plant status")
plt.ylabel("Number of plants")
plt.title("Number of plants in each status category")
plt.tight_layout()
plt.show()


## 21. Box plot for group comparison

A box plot summarises a numerical distribution within each category.

- the line inside the box is the median;
- the box covers the middle 50% of observations;
- whiskers extend beyond the box;
- isolated points may indicate potential outliers.

A box plot does not display every observation equally clearly, so interpretation should consider group size and overlap.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=clean_data,
    x="treatment",
    y="biomass_g",
    hue="treatment",
    palette="Set3",
    legend=False
)
plt.xlabel("Nitrogen treatment")
plt.ylabel("Biomass (g)")
plt.title("Biomass distribution under each treatment")
plt.tight_layout()
plt.show()


The graph can suggest treatment differences, but it does not by itself establish whether the differences are statistically convincing. Lecture 8 addresses that question.

## 22. Scatter plot for two numerical variables

Each point represents one plant. The horizontal coordinate gives gene expression; the vertical coordinate gives biomass.

`hue="treatment"` assigns colours according to treatment.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=clean_data,
    x="gene_expression",
    y="biomass_g",
    hue="treatment",
    style="species",
    s=80
)
plt.xlabel("Relative gene expression")
plt.ylabel("Biomass (g)")
plt.title("Gene expression and plant biomass")
plt.tight_layout()
plt.show()


When interpreting the scatter plot, consider:

1. **direction**: do larger x-values tend to accompany larger or smaller y-values?
2. **strength**: are points close to a clear pattern or widely scattered?
3. **groups**: do colours or marker styles occupy different regions?
4. **unusual observations**: does any point lie far from the others?

Association does not automatically prove causation.

## 23. Save the cleaned dataset

`to_csv()` writes the cleaned DataFrame to a new file. The raw file is preserved.

`index=False` prevents pandas from adding its row index as an extra CSV column.

In [ ]:
clean_data.to_csv(
    "SBI812_plant_growth_cleaned.csv",
    index=False
)


## 24. Summary

A reliable exploratory workflow is:

1. import the file;
2. inspect its first rows, dimensions, columns and data types;
3. detect missing, inconsistent and duplicated data;
4. clean using documented decisions;
5. select, filter and sort observations;
6. calculate numerical and categorical summaries;
7. group observations according to the research design;
8. choose graphs that match the variable types; and
9. interpret what the result means in the experiment.

### Practice activities

1. Display the last six rows of `clean_data`.
2. Select only Bean plants receiving Low_N treatment.
3. Sort those plants from the smallest to largest chlorophyll index.
4. Calculate mean chlorophyll for each species–treatment combination.
5. Create a box plot comparing chlorophyll among treatments.
6. Create a scatter plot of chlorophyll index against biomass, coloured by species.
7. Write three sentences interpreting one of the figures. Mention the visible pattern, group overlap and one limitation.